# 7.2 Coding Practice: Sentiment Analysis

This notebook compares two approaches to IMDb sentiment classification:
- a lexicon-based / rule-based method using VADER
- a supervised method using TF-IDF + Logistic Regression

We will focus on how the methods behave on real review text and why sentiment is often harder than it looks.

**Goal:** interpret model behavior before treating any prediction as automatically correct.

## Open in Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/mynewbook/blob/master/module7/week07_sentiment_practice.ipynb)

GitHub notebook: [module7/week07_sentiment_practice.ipynb](https://github.com/obscrivn/mynewbook/blob/master/module7/week07_sentiment_practice.ipynb)

In [ ]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import movie_reviews
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

nltk.download('movie_reviews', quiet=True)
nltk.download('vader_lexicon', quiet=True)

rows = []
for fileid in movie_reviews.fileids():
    label = 'positive' if fileid.startswith('pos/') else 'negative'
    rows.append({'review': movie_reviews.raw(fileid), 'label': label})

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Loaded {len(df)} movie reviews from the NLTK corpus.')
df.head()


## Part 1: Explore the sentiment task

Before fitting any model, inspect a few positive and negative reviews. Which reviews seem easy to classify, and which look more ambiguous or difficult?

In [ ]:
pd.set_option('display.max_colwidth', 180)
positive_examples = df[df['label'] == 'positive'].head(3)
negative_examples = df[df['label'] == 'negative'].head(3)

example_df = pd.concat([positive_examples, negative_examples], ignore_index=True)
example_df[['label', 'review']]


### Prediction check

Before running a sentiment model, which review do you expect to be easiest to classify? Which one might be hardest, and why?

## Part 2: Lexicon-based / rule-based sentiment with VADER

VADER uses a sentiment lexicon plus rules for punctuation, capitalization, and negation. It does not learn a model from labels; instead, it scores words and phrases based on their prior sentiment.

This is a good example of a lexicon-based method that is lightweight and easy to inspect.


In [ ]:
analyzer = SentimentIntensityAnalyzer()

def vader_scores(review):
    scores = analyzer.polarity_scores(review)
    return {
        'neg': scores['neg'],
        'neu': scores['neu'],
        'pos': scores['pos'],
        'compound': scores['compound'],
    }

vader_df = df.copy()
vader_df[['neg', 'neu', 'pos', 'compound']] = pd.DataFrame(vader_df['review'].apply(vader_scores).tolist())

# TODO: Complete the rule below so reviews with compound >= 0.05 are labeled positive,
# compound <= -0.05 are labeled negative, and neutral or borderline cases are assigned
# based on the sign of the score.
vader_df['vader_label'] = vader_df['compound'].apply(
    lambda score: 'positive' if score >= 0.05 else 'negative' if score <= -0.05 else ('positive' if score >= 0 else 'negative')
)

vader_df[['review', 'label', 'neg', 'neu', 'pos', 'compound', 'vader_label']].head(5)


### Interpretation questions

- What does the `compound` score represent?
- Find one review that the lexicon method labeled correctly and explain why it likely worked.
- Find one review it labeled incorrectly. What language may have confused it?
- Which kinds of sentiment are hard for a lexicon-based method?

> TODO: Compare the raw VADER score with the binary prediction and look for cases where the model is confident or uncertain.

In [ ]:
def vader_predict_label(review):
    scores = analyzer.polarity_scores(review)
    compound = scores['compound']
    if compound >= 0.05:
        return 'positive'
    if compound <= -0.05:
        return 'negative'
    return 'positive' if compound >= 0 else 'negative'

vader_df['vader_label'] = vader_df['review'].apply(vader_predict_label)
vader_df[['label', 'vader_label', 'compound', 'review']].head(5)


## Part 3: Supervised sentiment classification

Now we compare the lexicon method with a simple supervised baseline using TF-IDF features and logistic regression. This keeps the task grounded in the same IMDb review data without redoing all of the earlier preprocessing work from scratch.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['review'],
    df['label'],
    test_size=0.25,
    random_state=42,
    stratify=df['label'],
)

X_train.shape, X_test.shape


In [ ]:
vader_test_predictions = X_test.apply(vader_predict_label)
vader_accuracy = accuracy_score(y_test, vader_test_predictions)
print(f'VADER accuracy (held-out set): {vader_accuracy:.3f}')
print(classification_report(y_test, vader_test_predictions, digits=3))
print(confusion_matrix(y_test, vader_test_predictions, labels=['negative', 'positive']))

test_predictions_df = pd.DataFrame({'review': X_test, 'label': y_test}).reset_index(drop=True)
test_predictions_df['vader_label'] = vader_test_predictions.to_numpy()
wrong_examples = test_predictions_df[test_predictions_df['label'] != test_predictions_df['vader_label']].head(5)
wrong_examples[['label', 'vader_label', 'review']]


In [ ]:
model = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2, max_features=5000)),
    ('logreg', LogisticRegression(max_iter=1000, random_state=42)),
])

model.fit(X_train, y_train)

train_predictions = model.predict(X_train)
train_accuracy = accuracy_score(y_train, train_predictions)
print(f'Train accuracy: {train_accuracy:.3f}')

predictions = model.predict(X_test)
print(f'Test accuracy: {accuracy_score(y_test, predictions):.3f}')
print(classification_report(y_test, predictions, digits=3))
print(confusion_matrix(y_test, predictions, labels=['negative', 'positive']))


### Inspect a few errors

Look at the reviews the supervised model predicted incorrectly. Why do these reviews seem harder than the others?

In [ ]:
test_df = pd.DataFrame({'review': X_test, 'label': y_test}).reset_index(drop=True)
test_df['model_prediction'] = predictions
errors = test_df[test_df['label'] != test_df['model_prediction']].head(5)
errors[['label', 'model_prediction', 'review']]


## Part 4: Compare the two approaches

Run both models on the same small set of selected reviews and compare their outputs directly.

For a fair comparison, both methods are evaluated on the same held-out review set.

In [ ]:
selected = test_df.sample(6, random_state=42).copy()
selected['vader_prediction'] = selected['review'].apply(lambda r: analyzer.polarity_scores(r)['compound'])
selected['vader_prediction'] = selected['vader_prediction'].apply(
    lambda score: 'positive' if score >= 0.05 else 'negative' if score <= -0.05 else ('positive' if score >= 0 else 'negative')
)
selected['model_prediction'] = model.predict(selected['review'])
selected[['review', 'label', 'vader_prediction', 'model_prediction']].reset_index(drop=True)


### Compare the predictions

- Where do both methods agree?
- Where do they disagree?
- Which approach seems more reliable on this IMDb sample?
- In what situations would a lexicon-based method still be useful?

## Part 5: Challenge examples

Before running the next examples, predict whether each one is likely to be easy or difficult for VADER and for the supervised model.

In [ ]:
challenge_reviews = [
    'This movie is not terrible, but it is definitely not great either.',
    'I loved the first half, but the ending was so disappointing that I almost left.',
    'The film was unbelievably bad in the most entertaining way.',
    'I did not hate it, but I would not recommend it to a friend.',
]
for review in challenge_reviews:
    scores = analyzer.polarity_scores(review)
    print('REVIEW:', review)
    print(scores)
    print('VADER label:', ('positive' if scores['compound'] >= 0.05 else 'negative' if scores['compound'] <= -0.05 else 'positive' if scores['compound'] >= 0 else 'negative'))
    print('Model label:', model.predict([review])[0])
    print('-' * 80)


## Wrap-up

A good sentiment model is not just about getting a high overall accuracy score. The goal is to understand why sentiment is tricky: negation, sarcasm, mixed emotion, and context all matter.